일단 기본 4개 노드 4개 원형 엣지 구성으로 감


In [ ]:
from qiskit import *
from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit.primitives import StatevectorEstimator 
from functools import partial

import torch
from torch.autograd import Function
import torch.nn as nn
import torch.nn.functional as F 
import torch.optim as optim

import numpy as np
import matplotlib.pyplot as plt
import quimb as qu

import random
import time
import secrets
from EGATE import *
from NNVQE_HEA_half_uni import *

In [ ]:
import os
def generate_random_hamiltonian_graph(N=5, num_data=20):
    # 현재 RNG state(시드 상태) 백업
    # old_torch_state = torch.random.get_rng_state()
    # old_numpy_state = np.random.get_state()
    # old_py_state = random.getstate()

    # 그래프만 "진짜 랜덤"으로 만들기 위해, 시드를 None 또는 시스템 시계로 세팅
    # torch.manual_seed(int(time.time() * 1e6) + os.getpid())
    # np.random.seed(int(time.time()))
    # random.seed(int(time.time()))

    A = np.linspace(-3.0,3.0,num_data)
    # 랜덤 그래프 생성 edges도 달라질 수 있어야함
    if N==2:
        edges = [(0,1)]
    else:
        edges = [(i, i+1) for i in range(N-1)]
        edges.append((N-1, 0))
    edge_features = {}
    rand_vec = torch.rand(3)  # 기본적으로 [0,1] -> [0,2] -> [-1,1]
    
    # 첫 번째, 두 번째 값을 1로 지정
    rand_vec[0] = 1.0
    rand_vec[1] = 1.0
    rand_vec[2] = 1.0
    for e in edges:
        # edge_features[e] = torch.randint(-1, 2, (3,), dtype=torch.float32)  # 👈 0 또는 1로 설정
        edge_features[e] = rand_vec
        
    # 그래프 생성 완료 후, 복원
    # torch.random.set_rng_state(old_torch_state)
    # np.random.set_state(old_numpy_state)
    # random.setstate(old_py_state)

    return edges, edge_features

In [ ]:
sum_of_energy = []
def train(n, d, edge_full_data, latent_data, latent_size, NN_shape, maxiter=1, lr=0.1, stddev=1.0, index=0):
    model = NN_MERA_Model( n, d, stddev, NN_shape, latent_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    decay_steps = 700
    decay_rate = 0.7
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: decay_rate ** (step // decay_steps)
    )
    
    for i in range(1, maxiter + 1):
        optimizer.zero_grad()
        total_energy = torch.tensor(0.0, dtype=torch.float32)
        for j in range(1):
            edge_data = [t.tolist() for t in edge_full_data[j].values()]
            edges = list(edge_full_data[j].keys())
            graph_latent = latent_data[j]
            # print(edges)
            # print(edge_data)
            # print(graph_latent)
            energy = model(edges, edge_data, graph_latent)
            total_energy += energy
        total_energy.backward()
        grad_param = QuantumCircuitFunction.grad_params_buffer

        optimizer.step()
        scheduler.step()
        sum_of_energy.append(total_energy.item())
        if i % 10 == 0:
            print(f"Epoch {i}, Total Energy: {total_energy.item()}")
    # index = secrets.randbelow(10)
    # print(index)
    # torch.save(model.state_dict(), "GAE_NN_VQE_HEA.pth")
    return grad_param



In [ ]:
# Define training parameters
n_list = [3,4,5,6,7,8,9]          # Number of qubits
d = [1,2,3,4,5,6,7]  
NN_shape = 20
stddev = 1.0
maxiter = 1

iter = 500
results_dict = {}  # 각 n마다 여러 번의 결과(스칼라)를 담을 딕셔너리
var_dict = {}
var_mean_list = []

# Start training
for n in n_list:
  result_for_this_n = []
  # print("qubit ",n)
  # H-graph latent vector 구성
  latent_data = []
  edge_full_data = []
  if __name__ == "__main__":
      torch.manual_seed(1)
      np.random.seed(1)
      random.seed(1)
      node_feats = one_hot_encoding_nodes(n)
      edges, edge_feats_dict = generate_random_hamiltonian_graph(n,1)
      # print(edge_feats_dict)
      edge_full_data.append(edge_feats_dict)
      # print("\n=== Training Graph AutoEncoder [EGAT Layer with λ splitting] ===")
      model = EGATEAutoEncoder(
          node_in_dim= n,
          edge_in_dim= 3,
          node_hidden_dim= n,  
          edge_hidden_dim= 3,
          num_layers=3,
          decoder_hidden_dim=16,
          lambda_param=0.5,  # Lambda
          num_edges=len(edges)
      )
      num_epochs = 500
      H_rec, E_rec, mse_list, graph_latent = train_single_graph(
        model, edges, edge_feats_dict, node_feats, num_epochs=num_epochs
      )
      # # print(mse_list)
      # print(f"[Final MSE] {mse_list[num_epochs-1]}")
      # print("[Encoder - Node Embedding]:", H_final.shape)
      # print(H_final)
      # print("[Encoder - Edge Embedding]:", E_final.shape)
      # print(E_final)
      # print("[Graph-level Latent]:", graph_latent.shape)
      # print(graph_latent)
      latent_data.append(graph_latent)
      print(latent_data)
      # print("[Edge Feature Original vs Reconstructed]")
      # for k, e in enumerate(edges):
      #   print(f"  Edge {e}: org={edge_feats_dict[e]}, rec={E_rec[k]}")
  latent_size = n + 3
  _, param_num = HEA({"params": np.zeros(1000),  "edges": None, "edge_data":None}, n, d[n-3], param_num=True)
  print("qubit: {}, # of parameter: {}".format(n, param_num))
  old_py_state = random.getstate()
  random.seed(None)

  # index = secrets.randbelow(param_num) 
  index = random.randrange(0, param_num) 
  random.setstate(old_py_state)
  for i in range(iter):
    a = train(n, d[n-3], edge_full_data, latent_data, latent_size, NN_shape, maxiter,0.1, stddev, index)
    # print(a)
    result_for_this_n.append(a)
  # print("gradient index",in/ex)
  # print(result_for_this_n)

  var_list_n=[]
  for j in range(param_num):
      temp_gar = []
      for i in range(iter):
          temp_gar.append(result_for_this_n[i][j])
      # print(temp_gar)
      # print(np.var(temp_gar, ddof=1))
      var_list_n.append(np.var(temp_gar, ddof=1))
      # print()
  # print(len(var_list_n))
  print(np.mean(var_list_n))

  var_mean_list.append(np.mean(var_list_n))
  var_dict[n] = var_list_n
  results_dict[n] = result_for_this_n

In [ ]:
print("GAE_var_avg = ", var_mean_list)

In [ ]:
print("GAE_var_dict = ", var_dict)

In [ ]:
print("GAE_var_mean_list = ", var_mean_list)

In [ ]:
# Define training parameters
n_list = [10]          # Number of qubits
d = [8]  
NN_shape = 20
stddev = 1.0
maxiter = 1

iter = 500
# results_dict = {}  # 각 n마다 여러 번의 결과(스칼라)를 담을 딕셔너리
# var_dict = {}
# var_mean_list = []

# Start training
for n in n_list:
  result_for_this_n = []
  # print("qubit ",n)
  # H-graph latent vector 구성
  latent_data = []
  edge_full_data = []
  if __name__ == "__main__":
      torch.manual_seed(1)
      np.random.seed(1)
      random.seed(1)
      node_feats = one_hot_encoding_nodes(n)
      edges, edge_feats_dict = generate_random_hamiltonian_graph(n,1)
      # print(edge_feats_dict)
      edge_full_data.append(edge_feats_dict)
      # print("\n=== Training Graph AutoEncoder [EGAT Layer with λ splitting] ===")
      model = EGATEAutoEncoder(
          node_in_dim= n,
          edge_in_dim= 3,
          node_hidden_dim= n,  
          edge_hidden_dim= 3,
          num_layers=3,
          decoder_hidden_dim=16,
          lambda_param=0.5,  # Lambda
          num_edges=len(edges)
      )
      num_epochs = 500
      H_rec, E_rec, mse_list, graph_latent = train_single_graph(
        model, edges, edge_feats_dict, node_feats, num_epochs=num_epochs
      )
      # # print(mse_list)
      # print(f"[Final MSE] {mse_list[num_epochs-1]}")
      # print("[Encoder - Node Embedding]:", H_final.shape)
      # print(H_final)
      # print("[Encoder - Edge Embedding]:", E_final.shape)
      # print(E_final)
      # print("[Graph-level Latent]:", graph_latent.shape)
      # print(graph_latent)
      latent_data.append(graph_latent)
      print(latent_data)
      # print("[Edge Feature Original vs Reconstructed]")
      # for k, e in enumerate(edges):
      #   print(f"  Edge {e}: org={edge_feats_dict[e]}, rec={E_rec[k]}")
  latent_size = n + 3
  _, param_num = HEA({"params": np.zeros(1000),  "edges": None, "edge_data":None}, n, d[n-10], param_num=True)
  print("qubit: {}, # of parameter: {}".format(n, param_num))
  old_py_state = random.getstate()
  random.seed(None)

  # index = secrets.randbelow(param_num) 
  index = random.randrange(0, param_num) 
  random.setstate(old_py_state)
  for i in range(iter):
    a = train(n, d[n-10], edge_full_data, latent_data, latent_size, NN_shape, maxiter,0.1, stddev, index)
    # print(a)
    result_for_this_n.append(a)
  # print("gradient index",in/ex)
  # print(result_for_this_n)

  var_list_n=[]
  for j in range(param_num):
      temp_gar = []
      for i in range(iter):
          temp_gar.append(result_for_this_n[i][j])
      # print(temp_gar)
      # print(np.var(temp_gar, ddof=1))
      var_list_n.append(np.var(temp_gar, ddof=1))
      # print()
  # print(len(var_list_n))
  print(np.mean(var_list_n))

  var_mean_list.append(np.mean(var_list_n))
  var_dict[n] = var_list_n
  results_dict[n] = result_for_this_n

In [ ]:
# 9보다 오히려 더 크게 나와서 다시 돌려보는 거임 -> 같은 수준임

n_list = [10]          # Number of qubits
d = [8]  
NN_shape = 20
stddev = 1.0
maxiter = 1

iter = 500
# results_dict = {}  # 각 n마다 여러 번의 결과(스칼라)를 담을 딕셔너리
# var_dict = {}
# var_mean_list = []

# Start training
for n in n_list:
  result_for_this_n = []
  # print("qubit ",n)
  # H-graph latent vector 구성
  latent_data = []
  edge_full_data = []
  if __name__ == "__main__":
      torch.manual_seed(1)
      np.random.seed(1)
      random.seed(1)
      node_feats = one_hot_encoding_nodes(n)
      edges, edge_feats_dict = generate_random_hamiltonian_graph(n,1)
      # print(edge_feats_dict)
      edge_full_data.append(edge_feats_dict)
      # print("\n=== Training Graph AutoEncoder [EGAT Layer with λ splitting] ===")
      model = EGATEAutoEncoder(
          node_in_dim= n,
          edge_in_dim= 3,
          node_hidden_dim= n,  
          edge_hidden_dim= 3,
          num_layers=3,
          decoder_hidden_dim=16,
          lambda_param=0.5,  # Lambda
          num_edges=len(edges)
      )
      num_epochs = 500
      H_rec, E_rec, mse_list, graph_latent = train_single_graph(
        model, edges, edge_feats_dict, node_feats, num_epochs=num_epochs
      )
      # # print(mse_list)
      # print(f"[Final MSE] {mse_list[num_epochs-1]}")
      # print("[Encoder - Node Embedding]:", H_final.shape)
      # print(H_final)
      # print("[Encoder - Edge Embedding]:", E_final.shape)
      # print(E_final)
      # print("[Graph-level Latent]:", graph_latent.shape)
      # print(graph_latent)
      latent_data.append(graph_latent)
      print(latent_data)
      # print("[Edge Feature Original vs Reconstructed]")
      # for k, e in enumerate(edges):
      #   print(f"  Edge {e}: org={edge_feats_dict[e]}, rec={E_rec[k]}")
  latent_size = n + 3
  _, param_num = HEA({"params": np.zeros(1000),  "edges": None, "edge_data":None}, n, d[n-10], param_num=True)
  print("qubit: {}, # of parameter: {}".format(n, param_num))
  old_py_state = random.getstate()
  random.seed(None)

  # index = secrets.randbelow(param_num) 
  index = random.randrange(0, param_num) 
  random.setstate(old_py_state)
  for i in range(iter):
    a = train(n, d[n-10], edge_full_data, latent_data, latent_size, NN_shape, maxiter,0.1, stddev, index)
    # print(a)
    result_for_this_n.append(a)
  # print("gradient index",in/ex)
  # print(result_for_this_n)

  var_list_n=[]
  for j in range(param_num):
      temp_gar = []
      for i in range(iter):
          temp_gar.append(result_for_this_n[i][j])
      # print(temp_gar)
      # print(np.var(temp_gar, ddof=1))
      var_list_n.append(np.var(temp_gar, ddof=1))
      # print()
  # print(len(var_list_n))
  print(np.mean(var_list_n))

#   var_mean_list.append(np.mean(var_list_n))

#   var_dict[n] = var_list_n
#   results_dict[n] = result_for_this_n

In [ ]:
# 예시 데이터 리스트 (여기 원하는 리스트로 바꾸면 됨!)

# 평균 계산
for i in results_dict.keys():
  # print(i)
  mean_val = np.mean(results_dict[i])
  plt.scatter(i, mean_val, color='red', s=100, label=f"Mean = {mean_val:.2f}", edgecolors='k')
  plt.scatter([i]*len(results_dict[i]), results_dict[i], alpha=0.5)



# 평균 (강조된 점)
# 시각화 세팅
# plt.xlim(-0.3, 0.3)
# plt.xticks([])
plt.ylabel("Value")
plt.title("Distribution (cloud) and Mean (dot)")
# plt.grid(True, linestyle='--', alpha=0.3)
# plt.legend()
plt.show()


In [ ]:
# 예시 데이터 리스트 (여기 원하는 리스트로 바꾸면 됨!)

# # 평균 계산
# for i in results_dict.keys():
#   # print(i)
#   var = np.var(results_dict[i], ddof=1)
#   plt.scatter(i, var, color='red', label=f"Var = {var:.2f}")
plt.plot(n_list,var_mean_list)


# 평균 (강조된 점)
# 시각화 세팅
# plt.xlim(-0.3, 0.3)
# plt.xticks([])
# plt.ylabel("Value")
# plt.title("Distribution (cloud) and variance (dot)")
# plt.grid(True, linestyle='--', alpha=0.3)
plt.legend()
plt.show()


In [ ]:

stds = [np.std(var_dict[n]) for n in n_list]

# Plotting
plt.figure(figsize=(8, 5))
plt.plot(n_list, var_mean_list, linewidth=2, label='Mean')
plt.fill_between(n_list, np.array(var_mean_list) - np.array(stds), np.array(var_mean_list) + np.array(stds), alpha=0.2)

xtick_labels = [f"{val}({val-2})" for val in n_list]
plt.xticks(n_list, xtick_labels)
plt.ylabel("var[grad.]")
plt.xlabel("# of qubit (ansatz depth)")
plt.title('GAE')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
GAE_var_dict =  {3: [0.5871494731366377, 0.12001734339455668, 0.5482902164519502, 0.5700133577231322, 0.1022857547981865, 0.5275138799409862, 0.4891806959435414, 0.10598763115033996, 0.5121335034194636, 0.23223968126057712, 0.21054717387182914, 0.25256291794865643, 0.6178138096423931, 0.5711392218361686, 0.577700090672797, 0.5989757261851794, 0.5691191902715129, 0.6159145352684441, 0.44400107714068693, 0.4881597697186842, 0.4607798272415955, 0.46283563354463253, 0.4285992827962318, 0.4265987977742832], 4: [0.4266537643215984, 0.09534887103327909, 0.4036145767470184, 0.42533213546341175, 0.09330539355798516, 0.4020602777677695, 0.36701838502549394, 0.08049407230307504, 0.3546466344385271, 0.44961725101927935, 0.0937448923291031, 0.4347856397064163, 0.226801281191461, 0.21232604827094256, 0.24446267457033224, 0.2211321243206067, 0.46023750102600736, 0.43671917726703907, 0.41206331167117366, 0.40471139005192946, 0.4315899995821436, 0.45033119583141756, 0.39673048759460927, 0.416284089937802, 0.34137278944654653, 0.29024447075172116, 0.3678161313040288, 0.2720482815134401, 0.33064496139772126, 0.2966408896032328, 0.370328188025362, 0.29209491650824354, 0.3620904831370291, 0.28060574644798963, 0.3302246855686802, 0.33795191894554755, 0.4027667453864446, 0.3850676268405972, 0.35062851180083704, 0.3519892396060077, 0.39106676274504304, 0.38516922773164247, 0.3537211359748618, 0.349458103453671, 0.3360940607286714, 0.3585922141111898, 0.31490572270209455, 0.3469113022249053, 0.31726513358495684, 0.3387396316074049, 0.3074414871027774, 0.31271186905288156], 5: [0.36674073317575223, 0.02238149536479295, 0.3758509867574123, 0.3534708864804818, 0.019543122067296793, 0.3480221471262593, 0.3847245343537959, 0.016496417052993134, 0.40558059535958485, 0.376932398000428, 0.016806687776020214, 0.38024518122451767, 0.35723592187579956, 0.019351047342315818, 0.3509808639060461, 0.06430548997364034, 0.05692651279773252, 0.04256151936580948, 0.062339892573062286, 0.06700494333394937, 0.45702919462707425, 0.4343094687620167, 0.4466311811136758, 0.43223776789161106, 0.47376308391565003, 0.4613157616927046, 0.45596263323266484, 0.4466015714631511, 0.45348127249203296, 0.4755777570225719, 0.35843941549769304, 0.17341983006385522, 0.3518998572650118, 0.1743752699543238, 0.3741993481410827, 0.152798136823251, 0.3741527038880547, 0.1885720378936957, 0.3288808703422688, 0.2121300155661562, 0.17740726425878398, 0.16280270422392545, 0.17592111709287603, 0.1811867124396614, 0.1856567228919039, 0.36148993053204576, 0.41361582623979987, 0.37842390104832063, 0.39572944611546185, 0.3871656777218341, 0.3941329616357583, 0.45586826743890535, 0.3611077861206477, 0.36801936287129233, 0.37354358411647587, 0.3316292065442938, 0.2835495991892083, 0.3343350212500193, 0.28340568660785853, 0.36000336589638904, 0.27656426765613473, 0.34788690423741675, 0.26874672045280845, 0.2935875489220691, 0.295699872514543, 0.2555808063884904, 0.20560297724455884, 0.18268157962738157, 0.2416977359563018, 0.20751643187351287, 0.2969960686421251, 0.2945234208092314, 0.30332116449492874, 0.2894985735897346, 0.2848804353622751, 0.31404293534695876, 0.2923540763707846, 0.2811371564587631, 0.297565669213203, 0.2635781380226675, 0.2982336469000359, 0.33463510442430355, 0.3373609847022879, 0.3377172691796907, 0.3357016268803271, 0.3861675390865563, 0.3220213339709881, 0.3475614693252868, 0.2776692200879747, 0.3611573212423784], 6: [0.3367777923701259, 0.03744456275506576, 0.34823547926192316, 0.32110724475603497, 0.04381655209406069, 0.3065565762986856, 0.30584503613521197, 0.040344944108812564, 0.3091812347668679, 0.31227317820458184, 0.057222046893996924, 0.3225950458582333, 0.3476846922968166, 0.03949708143562103, 0.3393889727878502, 0.3138064603189689, 0.03168211002350886, 0.2955612743756015, 0.10888324693368244, 0.10474982335566625, 0.10964926592753738, 0.13429775795179283, 0.10670745770551136, 0.11872245603511444, 0.3404616284348648, 0.3164287889870988, 0.3033403246246581, 0.3334617717879432, 0.3093032906789465, 0.34895712245298327, 0.3390256407337375, 0.3029253139053777, 0.32937438952322495, 0.35668924368366467, 0.3157198097528016, 0.3380811790165181, 0.3239730835267292, 0.19322285167409753, 0.2837118779421328, 0.1753034616544315, 0.2834872748869738, 0.18463071728311428, 0.27606042768360967, 0.19368410379430365, 0.3250433169136409, 0.1820244013263874, 0.28219583655510216, 0.19111518712039247, 0.23699760312268786, 0.19349204222062213, 0.18964298669755175, 0.20852346945930192, 0.1944534544506917, 0.2108413318740069, 0.3091460858187904, 0.30533516840738967, 0.28367207435050884, 0.32342092247612586, 0.27688708430644876, 0.31036904565507073, 0.3646639303381433, 0.3078512597175207, 0.3070392911563379, 0.35109294152167814, 0.2823361158303068, 0.31949945923562534, 0.27197265182692304, 0.24756231769293324, 0.2574150144600882, 0.2569982902876318, 0.24746186744130796, 0.25812735088433914, 0.259202921128911, 0.23972937622913568, 0.29396317159871255, 0.23695770437125468, 0.28368351711135925, 0.2437773072761012, 0.2536578302530856, 0.21934135341789934, 0.21472059855535255, 0.23657893203857835, 0.2427123288949029, 0.2332961022164589, 0.3002159819563981, 0.28619575387260743, 0.2569743334047132, 0.2986911196369164, 0.29240857796773023, 0.27818544834784914, 0.31194322420199155, 0.26858605434170496, 0.23726387902201335, 0.2863300607471133, 0.317346983358798, 0.3044078458424494, 0.23188471300524457, 0.28585198680127905, 0.24889159116398574, 0.2516436370443942, 0.24718711377291083, 0.26330658848280225, 0.242422701206086, 0.27670800906438614, 0.2704839959933616, 0.2386609581874218, 0.2764730672243589, 0.28808679069490517, 0.21570069314188312, 0.19848549192312767, 0.21283245845568954, 0.2051624246472891, 0.24373202182266643, 0.2317217095065068, 0.21553358940095266, 0.24080018631427635, 0.2295787227046592, 0.2424649934596557, 0.20699732234492618, 0.2172066898726291, 0.20732129352964432, 0.23935717206553503, 0.22410382792079483, 0.2050572472137652, 0.22635786437558122, 0.22834210968110347, 0.2115268441494769, 0.32555799529439994, 0.2202596004520757, 0.305117070647861, 0.24486746690939712, 0.31118015373024216, 0.26944645043622933, 0.27105535867436853, 0.243504843766746, 0.26766982670789197, 0.2423353969384257, 0.31084106743322165], 7: [0.3277193054497268, 0.031136199826785945, 0.3237367871066749, 0.32832755686895376, 0.05030333884739463, 0.32625410548574346, 0.3220442008731372, 0.030510828107766966, 0.3227062140523421, 0.2829512502211579, 0.05379900633693828, 0.2864069671025184, 0.33516456693153895, 0.03023863832033326, 0.33054708982418063, 0.2952573934237381, 0.043136511839838486, 0.3027047195373717, 0.28286629284565695, 0.03924124234875051, 0.2878981921669187, 0.10821780868869668, 0.10855725591327915, 0.11557621720168024, 0.10606370577770005, 0.10622472717245714, 0.11696393037742726, 0.09436325772387687, 0.2855132849201026, 0.3068306782216737, 0.3477985165531808, 0.3172029892500838, 0.3309303720902859, 0.2838261043136072, 0.32292490968430293, 0.30148919985783745, 0.3361963191044994, 0.3194440742409967, 0.2860101818695123, 0.3314946096996726, 0.28803311764510076, 0.2997703491809246, 0.2883796223824213, 0.13457788606826593, 0.3000554545029122, 0.17140126334928318, 0.31011297401473387, 0.14137227266233876, 0.28626490771262947, 0.15973884794088697, 0.3020078123711703, 0.13645703359269046, 0.2724922150218809, 0.16036491568593886, 0.2796948838722858, 0.13474857387815914, 0.1717660614754353, 0.17160637092421355, 0.17970307713758524, 0.17951784705682053, 0.1750674242508241, 0.17243009195094847, 0.1512499113195399, 0.2799255287046851, 0.29804801227934236, 0.35113972569246577, 0.29319095405450474, 0.3028517925852446, 0.28063579419258616, 0.31038603570964723, 0.2837688273036904, 0.3133822041521496, 0.31406010247057037, 0.26202145927682907, 0.3198880407842494, 0.3047023271741362, 0.2796027231092526, 0.293593698685836, 0.21103625916319424, 0.26747926232029995, 0.22564101850668364, 0.27924260062137535, 0.22927445155175472, 0.2790727267371582, 0.2001694175448175, 0.28267254672187325, 0.18739926020856934, 0.27216150784391174, 0.20938421171099744, 0.2890060535447702, 0.18984978630324512, 0.2070431220126429, 0.2052954701120803, 0.22581628268239742, 0.21494569263818195, 0.2265626548224284, 0.20468800379564678, 0.1973998590251532, 0.24688681343034777, 0.27650444048552714, 0.2945905983495628, 0.28610602153291687, 0.25745129666908273, 0.2660420333106311, 0.28934698064949294, 0.25738238995435014, 0.3373750964437578, 0.28809606729086146, 0.251872516238467, 0.2921491958810682, 0.2746351992308911, 0.22044519241887406, 0.26283863940319613, 0.2622645989587622, 0.26373652468593545, 0.2658119699611814, 0.28464749111069987, 0.25345946616734155, 0.27118801963930633, 0.2577365385599072, 0.2655183130067117, 0.21494323299386825, 0.2578986456662256, 0.24823913907114417, 0.280706515737004, 0.2648988569209066, 0.22094473134367967, 0.2311879412432035, 0.22624120301657033, 0.19031304088827072, 0.22760538417579884, 0.23206034624216382, 0.20276450735527404, 0.23272777102835734, 0.23875118584690258, 0.24696599140161532, 0.23379826891906139, 0.23324311010535176, 0.23818366000023752, 0.2562583256867658, 0.24354927303010782, 0.2822640429402262, 0.26150444038862986, 0.2571682209316965, 0.2693742193868799, 0.2432463907941889, 0.25298120538446295, 0.24651334333197722, 0.3112239474811142, 0.24933393656136857, 0.27298649918918294, 0.243717580489164, 0.2705191008937504, 0.2412261660994164, 0.27769217008311936, 0.261865449886597, 0.26373242124756463, 0.2564334665365889, 0.2715331110472553, 0.25985984866929723, 0.2806003836848662, 0.19162800715074604, 0.20429114911354557, 0.23260118370315772, 0.19094404961166578, 0.2043785248503467, 0.2173418370403265, 0.20872163123880477, 0.19238872950074312, 0.14971307682685048, 0.20748659818502646, 0.1942752420927288, 0.18591267601780093, 0.1820436843810532, 0.23112705060444907, 0.20038308860520146, 0.20796930561929713, 0.20875962011530186, 0.21665899185908166, 0.24235280004843568, 0.22896161628714437, 0.21173891402005704, 0.25474723001315713, 0.3092695187834273, 0.22793988112753677, 0.3185111947097863, 0.233321997831564, 0.3195196793929322, 0.22295226946786936, 0.33642337441495807, 0.2216464088611875, 0.30075864474444136, 0.262030589665772, 0.3034692344716357, 0.24592100649728274, 0.3072349939674213], 8: [0.2907099771416506, 0.028687495290189685, 0.28989983213949994, 0.29399513990654, 0.03551087359173095, 0.2946227840509672, 0.2857883437375974, 0.030454921750414917, 0.2936693003628346, 0.2750475516883957, 0.026885247621892712, 0.2763307867253836, 0.2934171296178735, 0.0289512201429064, 0.28353693552647846, 0.3284347051186414, 0.04093528832688773, 0.29861940928203345, 0.3114476780712762, 0.04038364663805523, 0.30671590806840526, 0.34482016877826527, 0.03896834005033615, 0.3350609495079079, 0.09789103371776622, 0.08509746722120111, 0.09370112379634422, 0.08241937813924835, 0.1007841738835413, 0.10792434243188793, 0.09368838989024804, 0.09067801992307217, 0.29852643274129725, 0.3148299753565068, 0.3033286945227065, 0.2805416122077117, 0.2752963021484781, 0.2837398062616452, 0.29817407930339257, 0.26914441584432985, 0.29091956809156394, 0.3412229837999795, 0.3115260419755789, 0.31072230313407434, 0.2614340994834488, 0.28059491908318135, 0.2866460943092353, 0.2725801707409172, 0.2828996171196676, 0.13724579144299942, 0.28097950338521355, 0.15123707788132384, 0.27233645067579476, 0.13731157563293686, 0.26110636457553255, 0.11848508894882974, 0.2672143245962447, 0.11616714316618927, 0.2925532326390285, 0.14428496025673912, 0.30197119247468596, 0.12203113136451324, 0.3179852103203072, 0.14021681741904543, 0.17603477448735486, 0.15831885260177966, 0.1682205064941083, 0.14632571153333243, 0.17152956685032894, 0.18475371095463516, 0.17096042389788554, 0.15487141535699245, 0.2717647319168629, 0.31143749615498595, 0.3103491803002723, 0.2300367919812762, 0.28082262384024465, 0.2760510632557605, 0.3281209797355912, 0.2630553143288282, 0.2933022622236523, 0.3232208282231415, 0.26272787293851774, 0.2630624150567975, 0.26348817485934173, 0.2669191494533199, 0.2799739901893053, 0.2630563348146793, 0.23107300044380527, 0.18219260309772128, 0.24879011384322244, 0.18420249297898197, 0.24143011217479454, 0.17917629345153224, 0.2338339013759623, 0.1789941430660113, 0.25457782792989936, 0.1730019264290961, 0.26610810491985093, 0.18162161274665148, 0.29613513394384433, 0.18435613999258335, 0.30598927185514, 0.18167213753803496, 0.20007211711724399, 0.18053223657032716, 0.1907978108006659, 0.193626400371449, 0.19913898377053607, 0.2088028265063805, 0.18524145150653618, 0.21137124894498732, 0.2690670902626146, 0.29705017128577105, 0.2779881179718092, 0.23847969331891347, 0.2754727347850738, 0.2701183633515777, 0.3060299483699944, 0.2594994853186541, 0.2674321650729764, 0.27035951987941187, 0.2586813372963594, 0.2447216880807526, 0.2519979552574186, 0.23890460640154232, 0.2796079541073529, 0.2570076724338404, 0.23834643902569183, 0.2131047766515865, 0.2110689420707831, 0.21150907405940503, 0.2272954259183298, 0.19864450585832089, 0.2439488785295007, 0.198457150093767, 0.22953386727275063, 0.20950552809541598, 0.23335213822714168, 0.20402351198217103, 0.2531690673442136, 0.2125897420401582, 0.3022181081681103, 0.20098803779193733, 0.20243862395862922, 0.18498792814431447, 0.17995681736641544, 0.20093311930949045, 0.2004252059548891, 0.21867233270013062, 0.20653865609706965, 0.19006441845479025, 0.23798317460683366, 0.2727779632606468, 0.2697875875073288, 0.23378954930202198, 0.24962254646791165, 0.2658210774422713, 0.314405188627928, 0.26080073001397025, 0.26676592640874486, 0.2513994905777896, 0.25713046839291015, 0.24932553758171674, 0.24795718183574, 0.2579717332531417, 0.27349005884122507, 0.2562355069029456, 0.2158334206079476, 0.24314599225038772, 0.21771939991168476, 0.23934042448517853, 0.22748971420863479, 0.23036910086074336, 0.21842888164476248, 0.2394553303366404, 0.22473191410773857, 0.2154776063505987, 0.25389187150022036, 0.22356502431361047, 0.25119896786891, 0.21203347094498542, 0.2631060171553556, 0.21264538404264385, 0.19124740090678066, 0.2092862491208416, 0.1928601232547023, 0.18056472171582297, 0.20531988897448833, 0.22792570351882974, 0.18536206382674367, 0.20150592506303736, 0.23335331990042077, 0.25628711029181644, 0.2577703440286625, 0.20694703571273126, 0.23889324981530724, 0.2448080007350866, 0.2742258339054771, 0.22998393177266213, 0.24383443256492893, 0.24864913845033812, 0.23085922229808706, 0.24352270048584804, 0.246068344274246, 0.22835173411060145, 0.23338762096390622, 0.24500063128576371, 0.20345160443106552, 0.2893671312202346, 0.22797175975526313, 0.2819003886869137, 0.2282507908112602, 0.28283017015153833, 0.2276020741113348, 0.2599606760814038, 0.21199657990032403, 0.24965007224462926, 0.2530593911123332, 0.2114737400881239, 0.23335842783229593, 0.2502397039326308, 0.24019683098560943, 0.24674562571882178, 0.18123028658979878, 0.18988781500404597, 0.1863378528290439, 0.17406496023181312, 0.19266500104446913, 0.2029546441652515, 0.18278068243978401, 0.19230430595955217, 0.1826863431930146, 0.17825676617042854, 0.21213721606299926, 0.18528687471639618, 0.18180594921112775, 0.20001456938301615, 0.19757437136200218, 0.19635591019042697, 0.20791075866124945, 0.17174697016854873, 0.1866466135545385, 0.17150674776865701, 0.19737738054075335, 0.19615405631724855, 0.19272709816984296, 0.2105601602492714, 0.21899661365064815, 0.30110332643060517, 0.22237925108255463, 0.31316387411027596, 0.21169323940042503, 0.32579303476487825, 0.21670159319873789, 0.27222603474823537, 0.2001776130191839, 0.2641031405248248, 0.22739216817939809, 0.27226182650617, 0.23139546612549722, 0.26481563235512534, 0.23631980057123558, 0.261708053206108], 9: [0.2786767186680513, 0.030041071190353978, 0.26777840539433273, 0.2679339169903142, 0.03207538896122128, 0.2770974258236292, 0.31098759591266645, 0.030775681436680028, 0.2963048658940747, 0.2670644911998248, 0.029852110997813378, 0.2680707796045028, 0.24966987785724407, 0.02640635168547389, 0.24638298109504664, 0.2766890066433294, 0.026802983742499818, 0.27790372872557206, 0.2396747349168169, 0.038225737582900626, 0.23760409712124111, 0.2961363717507187, 0.03743274645377813, 0.3010656267605009, 0.2766050680849461, 0.028697211778784, 0.27144896286235176, 0.09246178650248695, 0.08237175649340484, 0.07732049200586505, 0.0749633859760634, 0.08295424557090145, 0.09363669889478235, 0.1009057069207261, 0.0947692554039775, 0.08490134223965759, 0.2747822632133793, 0.24556938134504466, 0.2807425816101772, 0.26176772207615817, 0.2926974032592604, 0.29390390020415913, 0.2460561463943431, 0.28584116781450536, 0.2678539080012635, 0.2831214050215613, 0.233109637200416, 0.2824736828523817, 0.2661522559515106, 0.2991612445768724, 0.3044196068807853, 0.26666907926568145, 0.29914162730965355, 0.2959554809152484, 0.28676671173110835, 0.15112946741157418, 0.2508571686390468, 0.1285987846640339, 0.2916150291647035, 0.12770205321307412, 0.23665004595300884, 0.10833959924925114, 0.22333569705507006, 0.11223920200950596, 0.27284479779798065, 0.12155888918753777, 0.24807188692367987, 0.13693573947679372, 0.2658918118210439, 0.1167877505558528, 0.2560766488282168, 0.10779086823244276, 0.1650505694467087, 0.1483556524112103, 0.13169480218263016, 0.1499080606085827, 0.1533813210141323, 0.146638775987212, 0.16610975395495295, 0.15396000695271855, 0.14885137474413576, 0.2620987619392356, 0.24192264829229654, 0.29356036115360473, 0.2508333611936873, 0.300060356764349, 0.3154498386056271, 0.245888626195596, 0.2613497847421488, 0.2709593367333424, 0.2748066619631582, 0.23094437973066684, 0.27069635608990383, 0.2654705816368731, 0.2601454191631337, 0.2780718088397486, 0.24863994870489148, 0.27914230971278853, 0.2743812807988981, 0.2634428649171487, 0.18738209242815654, 0.2393043802711722, 0.1587391452021643, 0.2674530609298449, 0.15403285902292552, 0.2248090613144536, 0.16006903422582203, 0.20980533782653743, 0.16207024035444648, 0.278882057960172, 0.16813718749787276, 0.23369498111801174, 0.16882578827430567, 0.2604428224966661, 0.17985013952007503, 0.22189794283351696, 0.14971068106409569, 0.1857344528455291, 0.18411258773254338, 0.17304188999382952, 0.17966254296912756, 0.17398591206280217, 0.19865906701986552, 0.20026947189803265, 0.1808070651255228, 0.1793477853465387, 0.2332099129376576, 0.23730482681105894, 0.2605282764979225, 0.2481115949239915, 0.2983875020012569, 0.26825427381683975, 0.21809520119626383, 0.2486225418679163, 0.24690194585241795, 0.24429397226680297, 0.24772653031695124, 0.2490645405315176, 0.2552397758719382, 0.26221808134236146, 0.2852136996011669, 0.22880571210730802, 0.2574782666544717, 0.2609646444043968, 0.23038112822434934, 0.20949129675883915, 0.22642320497646434, 0.1744852742355176, 0.22697580076839818, 0.18032970362575276, 0.2276904273402353, 0.1844874163288368, 0.21208581910447052, 0.19325186677438264, 0.24330297637950005, 0.1913051478388996, 0.23830475783785837, 0.17984810105338228, 0.25131816544685953, 0.19684823380217528, 0.2116472616822901, 0.19096873500173042, 0.18855584965215824, 0.1796958706531587, 0.18471486973340737, 0.18416477196665565, 0.18830949304552463, 0.2029624807206458, 0.21314542946049814, 0.1892891993479573, 0.19160722387037568, 0.22994138006732373, 0.23687862494690515, 0.2365969734181159, 0.2249124143373949, 0.2502064143698025, 0.2488443178175546, 0.23272889244394024, 0.2585191966664469, 0.23382598396146043, 0.2767604453879834, 0.24112332876092266, 0.2390479106589191, 0.23466107280878667, 0.24343373214348266, 0.2493070318246015, 0.2524789432165624, 0.24731046370564996, 0.25997102927007615, 0.23233621575097047, 0.2368519206910828, 0.2332976216612373, 0.19740343364305143, 0.21677891706589644, 0.18942751753682133, 0.23421888885086029, 0.18564365774031621, 0.19476639114909067, 0.21963286214349517, 0.23262362494081684, 0.2182585387624473, 0.22203149299660915, 0.21667135667724216, 0.22884464050362746, 0.20666261451798196, 0.22019006767209925, 0.20397703422444158, 0.18794734826277612, 0.19235324040217106, 0.1911379597944737, 0.19295405878636307, 0.2105221875636889, 0.21152160590365118, 0.20088158393391659, 0.1818496824942799, 0.1872451480536679, 0.2304426627268978, 0.22047986274213274, 0.22963403352275114, 0.22316427783588583, 0.22999378535324322, 0.22582883470037038, 0.23121518447318748, 0.24224881800846348, 0.20997664750883857, 0.258975604109884, 0.26095140824009266, 0.24044993648094673, 0.22272193072012764, 0.2281874105177209, 0.22299602127128942, 0.23608461119732746, 0.24991697096719712, 0.23077159645747927, 0.211134058855184, 0.23257896370823633, 0.22844384993239092, 0.21824302007632498, 0.2204257090818313, 0.19647208429455612, 0.210596093112554, 0.202567526454779, 0.19298243153120584, 0.2499177506513801, 0.22595544958067368, 0.23524649757403418, 0.22937490587783893, 0.22419048289841353, 0.21685973604363437, 0.22372126158332792, 0.21639485444353246, 0.23256419406193635, 0.19266589477149731, 0.1848864877877146, 0.18822878285240324, 0.18672925963116183, 0.19383603232943586, 0.19048575654911548, 0.18479083580526762, 0.1884235528147793, 0.1633839555365524, 0.22136019291883294, 0.1910609188406355, 0.2085948956025366, 0.19839101412184312, 0.19481146100369612, 0.19873657371070727, 0.20543962374712638, 0.20150949318929534, 0.2180500194372359, 0.23322390561480769, 0.2249313257074689, 0.20080314531083437, 0.20405633564379544, 0.20444023496322045, 0.2169506879754205, 0.20515332545640566, 0.21041099841303787, 0.2158887989009122, 0.23537000883548992, 0.24876891654512845, 0.2139769140116742, 0.2373116096854087, 0.2071922472054764, 0.224467912722602, 0.20643854907755868, 0.23281694598817676, 0.2023096548277014, 0.2624303596595107, 0.21462494229606022, 0.25865378533541983, 0.20630368181411546, 0.2251422448278893, 0.21473494721878075, 0.24605322599631438, 0.21860739796423467, 0.2303204393114071, 0.17547820974856213, 0.17818712237576614, 0.17122281367604492, 0.15743088276877187, 0.1797688312826841, 0.16424165638604452, 0.1809289886692376, 0.16826141962161348, 0.15858624630046253, 0.16980461818238968, 0.14443829962387067, 0.1761152977861527, 0.16945605559128368, 0.1749922556998832, 0.1501997244003716, 0.1534984722361369, 0.18582351937204433, 0.18081155028379758, 0.16687436779890308, 0.18520147207165158, 0.17030108523628737, 0.15174311379627883, 0.15984606168740648, 0.17246239852669473, 0.16296714577754703, 0.17506823358652873, 0.18797636072058177, 0.20654648671070375, 0.2468725147601869, 0.2087554256488599, 0.23926414740646995, 0.19971900918272062, 0.24623270646565593, 0.22230159865820737, 0.24449522658354148, 0.19736280523507008, 0.25577295837926506, 0.2165170244862843, 0.2661205191623056, 0.21697088505303685, 0.25672444782023096, 0.18918332802682314, 0.25838666148658584, 0.21562111299440753, 0.27333312223091955], 10: [0.30601679065525367, 0.033599469439004125, 0.2905046566600122, 0.31948922189982015, 0.03380539297874947, 0.3138977386046046, 0.32233706526324585, 0.032926249683915, 0.3151157967636957, 0.32866045947002176, 0.023396520234226267, 0.32862813617257375, 0.39283922629043627, 0.018185341126954163, 0.3958353558060087, 0.32718542136028667, 0.03163091423327495, 0.3177741712611048, 0.2727182713910971, 0.027321679409925977, 0.26611391003097984, 0.28207764540287467, 0.0237574410182011, 0.2931209015461707, 0.30061059645363153, 0.030202901840624838, 0.30684799402294055, 0.3096552181545093, 0.029043955244266322, 0.2947844152971419, 0.0726439569108995, 0.08415789906662047, 0.07869792505062306, 0.06214221739163995, 0.06421103807781914, 0.08593959126529735, 0.07736736227871437, 0.08271298357300597, 0.08469602314799399, 0.08252485226364445, 0.27125737128342614, 0.3162237102503256, 0.30564080966679735, 0.32126254941379895, 0.33777537241503264, 0.3365167679481165, 0.2666914212003374, 0.3162192355397731, 0.3088257720967562, 0.28655282617547395, 0.2679451843394072, 0.30522720898142647, 0.2950138711207087, 0.32306187061629293, 0.3289961663336739, 0.3125813916962572, 0.2598626951542734, 0.3403286559055615, 0.3317831898745034, 0.2874964425742086, 0.2974474281555362, 0.12502631261182331, 0.2896565354477003, 0.13210805847502874, 0.2786149141285419, 0.15652719939733103, 0.31572177631885034, 0.11203049649641425, 0.3299152830070744, 0.09888024947511044, 0.3001659063254237, 0.12786963009529662, 0.2555291010388625, 0.11347244230135733, 0.2555816412325227, 0.11288582292155498, 0.28179872394867705, 0.12807413481832505, 0.2669070942787633, 0.13592257923556383, 0.14434315765905112, 0.15089201995496423, 0.15421471365313413, 0.14453613957930456, 0.12601155081133883, 0.13897391190728348, 0.15254529098206884, 0.1318050875136617, 0.16211755824116716, 0.15539378051573016, 0.2953750009584119, 0.30053807166519697, 0.3133562096197212, 0.32131224848381407, 0.32365445949846755, 0.3314133198870843, 0.2643538583528645, 0.3136060003957065, 0.3101996699469655, 0.3135619392045297, 0.2565618079449953, 0.28526029500651107, 0.2926455064177003, 0.3304428795266806, 0.30679702296183714, 0.3048706344995408, 0.27359000966423375, 0.32455592500482744, 0.32962630849946467, 0.2856655577374675, 0.27480316666164833, 0.1791231701443751, 0.29445962356145594, 0.1717903709842536, 0.2681817545272063, 0.1992122774441192, 0.2775954194363792, 0.18736361403139576, 0.28298046367160223, 0.17061548295470225, 0.2957230479758101, 0.19120658418013353, 0.25494798886117964, 0.1541639902519257, 0.25214303755597334, 0.1456849444769936, 0.26646344209128164, 0.1685591453109088, 0.2799452251660966, 0.1810084672721848, 0.1712145013330605, 0.204604003791979, 0.19179682574246434, 0.20616393350544324, 0.18313617309268387, 0.19743081677552055, 0.18489540531577292, 0.17625628011363628, 0.1883920555289143, 0.17376149251867645, 0.3095057382240837, 0.30122607137301965, 0.3451060488546078, 0.31365551047116164, 0.30834030091919257, 0.3095352642164584, 0.27497761918376, 0.2877189006577698, 0.30075955176787633, 0.31249140310861323, 0.25603279923014527, 0.2575299152709823, 0.3080003720836368, 0.29302005270783266, 0.29684520070354353, 0.28837032799173345, 0.27634284781115154, 0.30328648618313386, 0.3352728648545378, 0.30294571630975725, 0.2668490326930216, 0.2067211265891911, 0.27179340544994846, 0.1873038918382487, 0.2657866875235049, 0.23711261250053764, 0.27843344627026073, 0.21659384114631383, 0.2952374249855107, 0.20116813248500715, 0.27761554932551885, 0.2161267198806754, 0.23482209608611948, 0.20649558260896042, 0.24758544692990306, 0.1833432730665727, 0.25417581092743335, 0.24115686526192828, 0.2682951519047619, 0.23344055497865163, 0.1833728486741721, 0.24148549080379705, 0.20442227065854598, 0.21432616416676914, 0.21053219802430373, 0.21853443907599635, 0.21450391215097298, 0.1972717728527679, 0.2319551515790629, 0.1985868952544663, 0.2865954393490085, 0.30343214763528503, 0.32233077547861677, 0.31094109218837584, 0.30102569712413446, 0.27111818550593253, 0.2655525278268468, 0.24845411126626277, 0.3036376029673849, 0.29089506740012094, 0.24641300843298106, 0.24905008891449326, 0.28436399967158216, 0.3026799835163315, 0.29349508986434675, 0.29513048234898215, 0.2610760900397551, 0.2618168965987274, 0.3533769694805446, 0.28877377767629625, 0.2401683798943058, 0.217381352357299, 0.2460088141190927, 0.2205859299236912, 0.23908219660070135, 0.24109359809858594, 0.2564597106343997, 0.2390385624678078, 0.2833593103946575, 0.23760209867360138, 0.26968094419651456, 0.24422038046273098, 0.24431431008699908, 0.21076288152778602, 0.23292835169994736, 0.23307939093764124, 0.23758865059160972, 0.26168645100149845, 0.24901366653571072, 0.2363039329594486, 0.202470982475135, 0.2097115492236077, 0.20300481079756666, 0.22668590047651307, 0.20671783529702323, 0.20401526662145839, 0.22224538264197696, 0.21494476555175995, 0.2371244685764871, 0.19635209148303576, 0.2791361726802738, 0.27067674112148943, 0.29631266524473304, 0.3020141072574648, 0.28937395121912957, 0.24753262842816842, 0.246557267434986, 0.25225414115444283, 0.2767188969236446, 0.28740573737821357, 0.27109153263335223, 0.28313237757381354, 0.2699413943642068, 0.3231516005617138, 0.26241864839296275, 0.2848263953705948, 0.2526147916799772, 0.27820985309666746, 0.2954696603346935, 0.2670293290240756, 0.23849300782967853, 0.2342044139632148, 0.24617648339263073, 0.22587977179492075, 0.2552527041644745, 0.26239471524271724, 0.24984812394118752, 0.2348235703758118, 0.2851739895311675, 0.24144215305246813, 0.26346995710193793, 0.24174656051836252, 0.2280382888761965, 0.2378589673537124, 0.24001798682684758, 0.24109412521977625, 0.23307369291184657, 0.2889279048879846, 0.2502201914022999, 0.2681809865279758, 0.1866298273183735, 0.22202670326202856, 0.201595500072098, 0.23792051230398312, 0.22591042108927262, 0.22306046041245514, 0.2336133182690429, 0.22158215344489735, 0.21525566385983846, 0.2086188936042684, 0.24872654663244803, 0.24945190766990252, 0.2525692625532782, 0.30028740310106966, 0.2503337480006983, 0.2246005530364865, 0.22174522662948415, 0.2340939003670182, 0.2638976592796245, 0.28923503385568683, 0.2501847861069852, 0.2317924594537602, 0.24082701941388365, 0.2781658238329049, 0.25493319632611977, 0.2664604875854112, 0.24704411160210488, 0.2680232378860256, 0.2728859021959386, 0.2659586877826799, 0.22643923491744436, 0.2565444542821912, 0.23764897292926293, 0.25106621526492323, 0.23235950042453868, 0.26750786593460113, 0.23069080764295694, 0.28118117164712203, 0.26450374163181206, 0.26980459484112435, 0.25748864200636773, 0.29049000163698646, 0.21905872624521894, 0.22419388974177828, 0.22442466327667387, 0.236040685704344, 0.2202234582715805, 0.29970737093757915, 0.2512074530995494, 0.2762798788241214, 0.18475968870520926, 0.22597375743822584, 0.2076930094226125, 0.24955710465471947, 0.2270637057380819, 0.2020686131675327, 0.22046944933776894, 0.21256353196236102, 0.1980108823900515, 0.19556035230874488, 0.20604271402593954, 0.21686130720769267, 0.23280790331656845, 0.23780756153218535, 0.22227109180189894, 0.2078234575897673, 0.20110189493418737, 0.21972762052110956, 0.2361057482491811, 0.25334673649317424, 0.24116779455312612, 0.23055843693071162, 0.21810723293599907, 0.24136804308185086, 0.23903502444748626, 0.2256103443875356, 0.2135524982077834, 0.2239668929668218, 0.2620183358622974, 0.2527555864716105, 0.2136709075287662, 0.305987053871682, 0.22736771667167668, 0.2913634856802117, 0.23281007567525114, 0.2776759827427743, 0.22661676197203656, 0.29825513409569143, 0.2434788313555766, 0.3008017136731279, 0.24666639401204724, 0.3034910803647839, 0.21551630340395186, 0.26901505084935956, 0.2542115073880888, 0.2626318024552967, 0.2251112460236251, 0.32539472769664274, 0.2433259315832313, 0.28354731883636164, 0.17892617011263434, 0.21085050109441442, 0.18806498188620666, 0.20925529497524384, 0.18497786112623837, 0.1658847534395716, 0.19320475854473118, 0.18591636017258234, 0.18329924309636697, 0.1896775432299937, 0.16930889796115664, 0.15626512601408138, 0.17121217100603964, 0.20277405635573068, 0.1876153853915542, 0.19864250505841619, 0.1553931324972863, 0.17582532447477034, 0.17437739834550875, 0.21423871587634147, 0.18804082687460832, 0.1997304266185368, 0.191038161185155, 0.18602392531922707, 0.19507481476370586, 0.19019293072698729, 0.1699144303110314, 0.19004845430849374, 0.1984502243536893, 0.22590977797010786, 0.23529494515890695, 0.3092135368442893, 0.22276225910740718, 0.28141033754257827, 0.24334804424842726, 0.2870863650471216, 0.20689750732703205, 0.3062940679793563, 0.22165824655815772, 0.3087258944078032, 0.2365958391567225, 0.32324360892776366, 0.20935514228713906, 0.29896522222434296, 0.21526961381817694, 0.26703957261858574, 0.218268863933964, 0.3107828185101976, 0.22774397129040774, 0.29995729305090296]}


In [ ]:
GAE_var_avg_poly_zero = []
n_list=[3,4,5,6,7,8,9,10]
for n in n_list:
    GAE_var_avg_poly_zero.append(GAE_var_dict[n][0])
print("GAE_var_avg_const_zero = ", GAE_var_avg_poly_zero)